## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [ ]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        '''self.W1 是第一层权重，形状是 [28*28, 100]，表示把一张 28x28 的图片展平成 784 维后，映射到  
        100 个隐藏单元。self.b1 是第一层偏置，形状是 [100]。self.W2 是第二层权重，形状是 [100, 10]，把隐藏层映射到 10 个类别；
        self.b2 是输出层偏置，形状是 [10]。'''
        ####################
        self.W1 = tf.Variable(tf.random.normal([28 * 28, 100], stddev=0.1))                                                   
        self.b1 = tf.Variable(tf.zeros([100]))                                                                                
        self.W2 = tf.Variable(tf.random.normal([100, 10], stddev=0.1))                                                        
        self.b2 = tf.Variable(tf.zeros([10]))


    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        '''第一步把输入图片从 [batch_size, 28, 28] 变成 [batch_size, 784]。
        第二步先做线性变换 xW1 + b1，得到隐藏层，再经过 ReLU 激活函数。tf.matmul(x, self.W1) 的结果形状是 [batch_size, 100]，加上 b1  时会自动广播。
        第三步把隐藏层映射到 10 维输出，得到 logits，形状是 [batch_size, 10]。'''
        ####################
        x = tf.reshape(x, [-1, 28 * 28])                                                                                      
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)                                                                      
        logits = tf.matmul(h1, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [7]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 0.6945653 ; accuracy 0.8259
epoch 1 : loss 0.69382674 ; accuracy 0.82603335
epoch 2 : loss 0.693091 ; accuracy 0.8261167
epoch 3 : loss 0.6923577 ; accuracy 0.8263
epoch 4 : loss 0.6916272 ; accuracy 0.82636666
epoch 5 : loss 0.6908992 ; accuracy 0.8265333
epoch 6 : loss 0.6901741 ; accuracy 0.82661664
epoch 7 : loss 0.68945146 ; accuracy 0.82673335
epoch 8 : loss 0.68873143 ; accuracy 0.8268833
epoch 9 : loss 0.688014 ; accuracy 0.82701665
epoch 10 : loss 0.6872991 ; accuracy 0.82713336
epoch 11 : loss 0.68658674 ; accuracy 0.8272333
epoch 12 : loss 0.68587697 ; accuracy 0.82733333
epoch 13 : loss 0.68516946 ; accuracy 0.82741666
epoch 14 : loss 0.6844646 ; accuracy 0.8275333
epoch 15 : loss 0.6837624 ; accuracy 0.82773334
epoch 16 : loss 0.6830626 ; accuracy 0.82785
epoch 17 : loss 0.68236536 ; accuracy 0.8279667
epoch 18 : loss 0.6816705 ; accuracy 0.82808334
epoch 19 : loss 0.6809781 ; accuracy 0.82818335
epoch 20 : loss 0.68028814 ; accuracy 0.8283333
epoch 21 : los